<a href="https://colab.research.google.com/github/SchwartzNU/DJ_schwartzlab/blob/master/PreprocessingRGC_Jul2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mat73
import mat73
data_dict = mat73.loadmat('/content/drive/MyDrive/RGC_Jul2026/newRGC_withnames.mat')

In [ ]:
import numpy as np
unique_names, counts = np.unique(data_dict['output']['cell_names'], return_counts=True)
etype_cell = []
for id in unique_names:

  idx = [i for i, x in enumerate(data_dict['output']['cell_names']) if x == id]

  etype_cell.append(data_dict['output']['cell_type_labels_numeric'][idx[0]])
unique_traces, counts_t = np.unique(data_dict['output']['cell_type_labels_numeric'], return_counts=True)


In [ ]:
nidx = []
all_len = len(data_dict['output']['cell_type_labels_numeric'])
for i in range(alllen):
    if data_dict['output']['raw_traces'][i].shape[0]>100000:
       nidx.append(i)

import numpy as np
newidx = np.setdiff1d(np.arange(all_len), nidx)
new_traces = [data_dict['output']['raw_traces'][i] for i in newidx]
new_celllabels = [data_dict['output']['cell_type_labels_numeric'][i] for i in newidx]
new_currents = [data_dict['output']['injected_currents'][i] for i in newidx]
new_clabels = [data_dict['output']['current_labels'][i] for i in newidx]
new_names = [data_dict['output']['cell_names'][i] for i in newidx]

selected_labels = [3,  5,  6, 11, 12, 13, 19, 23, 24, 30]
label_map = {label: idx for idx, label in enumerate(selected_labels)}


In [ ]:
!pip install ssqueezepy

In [ ]:
import torch
from ssqueezepy import ssq_cwt, extract_ridges
import os

def process_single_cell_subsample(trace_idx, trace, current_labels, injected_currents, cell_type, cellname):
    wavelet = ('gmw', {'beta': 20, 'gamma': 3})
    fs = 500000
    Tx, Wx, ssq_freqs, scales, *_ = ssq_cwt(trace, wavelet=wavelet, fs=fs)
    cwt_abs = np.abs(Wx)

    return trace_idx, cellname, cell_type, current_labels, injected_currents, cwt_abs[:,::100]

In [ ]:
from collections import defaultdict
from joblib import Parallel, delayed
def analyze_cell_types_parallel_verbose(output_struct, n_jobs=8):
    """Parallel version with joblib's built-in progress"""

    total_cells = len(output_struct['raw_traces'])
    print(f"Starting analysis of {total_cells} cells...")


    selected_labels = {3,  5,  6, 11, 12, 13, 19, 23, 24, 30} #these IDs depend on the specific dataset


    # Use verbose=10 to see joblib's progress
    all_results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(process_single_cell_subsample)(
            trace_idx,
            output_struct['raw_traces'][cell_idx],
            output_struct['current_labels'][cell_idx],
            output_struct['injected_currents'][cell_idx],
            output_struct['cell_type_labels_numeric'][cell_idx],
            output_struct['cell_names'][cell_idx]
        )
        for trace_idx in range(7117)
        if output_struct['cell_type_labels_numeric'][cell_idx] in selected_labels
    )

    print("Analysis complete! Grouping results...")

    # Group results by cell type
    #results_types = defaultdict(list)
    #results_clabels = defaultdict(list)
    #for cell_type, clabels, cwt_abs in all_results:
    #    results_types[int(cell_type)].append(cwt_abs)
        #results_clabels[int(clabels)].append(cwt_abs)

    return all_results#results_types #, results_clabels

In [ ]:
all_results = analyze_cell_types_parallel_verbose(output_struct)
results_types = defaultdict(list)
for cellname, cell_type, clabels, injected_currents, cwt_abs in all_results:
    tmplist = list()
    tmplist.append(cwt_abs)
    tmplist.append(cellname)
    tmplist.append(injected_currents)
    results_types[int(cell_type)].append(tmplist)